In [17]:
import torch
from torch import nn
from clearml import Task, Dataset
import json
import pandas as pd

In [3]:
with open("creds.json", "r") as lines:
    creds = json.load(lines)

Task.set_credentials(
    api_host=creds["api_server"],
    web_host=creds["web_server"],
    files_host=creds["files_server"],
    key=creds["access_key"],
    secret=creds["secret_key"]
)

In [19]:
# 1. Определяем класс SimpleNN

class SimpleNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(in_features=5, out_features=5)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(in_features=5, out_features=1)


    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x


# 2. Инициализация ClearML-задачи
task = Task.init(project_name="SimpleNN Project", task_name="Experiment #1")
config = {"learning_rate": 0.01, "batch_size": 2, "epochs": 20}
task.connect(config)


# 3. Создаём модель, оптимизатор и функцию потерь
model = SimpleNN().to(torch.device('cpu'))
optimizer = torch.optim.SGD(model.parameters(), lr=config["learning_rate"])
criterion = nn.MSELoss()


# 4. Генерируем случайные входные данные и метки
# torch.manual_seed(0)
# X = torch.randn(3, 5)
# y = torch.randn(3, 1)
dataset = Dataset.get(dataset_project="SimpleNN Project", dataset_name="synthetic_data") 
data_path = dataset.get_local_copy()  # ← здесь получится путь к папке с CSV
data_file_path = data_path + '/data.csv'
data_dataframe = pd.read_csv(data_file_path)
X = torch.Tensor(data_dataframe[["f{}".format(i+1) for i in range(5)]].values)
y = torch.Tensor(data_dataframe[["label"]].values)



# 5. Обучение модели
for epoch in range(config["epochs"]):
    outputs = model(X)
    loss = criterion(outputs, y)


    # Шаг обучения
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


    # Логируем loss с указанием итерации
    task.get_logger().report_scalar("Loss", "train", loss.item(), epoch)
    print(f"Epoch {epoch + 1}/{config['epochs']}, Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 1/20, Epoch 0, Loss: 0.3750
Epoch 2/20, Epoch 1, Loss: 0.3380
Epoch 3/20, Epoch 2, Loss: 0.3066
Epoch 4/20, Epoch 3, Loss: 0.2798
Epoch 5/20, Epoch 4, Loss: 0.2568
Epoch 6/20, Epoch 5, Loss: 0.2369
Epoch 7/20, Epoch 6, Loss: 0.2197
Epoch 8/20, Epoch 7, Loss: 0.2047
Epoch 9/20, Epoch 8, Loss: 0.1915
Epoch 10/20, Epoch 9, Loss: 0.1798
Epoch 11/20, Epoch 10, Loss: 0.1695
Epoch 12/20, Epoch 11, Loss: 0.1604
Epoch 13/20, Epoch 12, Loss: 0.1524
Epoch 14/20, Epoch 13, Loss: 0.1450
Epoch 15/20, Epoch 14, Loss: 0.1384
Epoch 16/20, Epoch 15, Loss: 0.1326
Epoch 17/20, Epoch 16, Loss: 0.1273
Epoch 18/20, Epoch 17, Loss: 0.1224
Epoch 19/20, Epoch 18, Loss: 0.1180
Epoch 20/20, Epoch 19, Loss: 0.1141


/var/folders/09/_l2m2cg560xbt2z696htd26h15_c_h/T/ipykernel_83178/2231447789.py:5: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  torch.Tensor(data_dataframe[["label"]].values)


tensor([[ 0.1227],
        [-0.5663],
        [ 0.3731]])

In [ ]:
# from clearml import Dataset


# dataset = Dataset.create(dataset_project="SimpleNN Project", dataset_name="synthetic_data")


# # Генерируем случайные данные как в прошлом задании
# torch.manual_seed(0)
# X = torch.randn(3, 5)   # форма [3, 5]
# y = torch.randn(3, 1)   # форма [3, 1]


# # Превращаем X и y в pandas.DataFrame
# # Сначала приводим к numpy, затем создаём DataFrame

# df = pd.DataFrame(
#     X.numpy(),
#     columns=[f"f{i+1}" for i in range(5)]   # f1, f2, f3, f4, f5
# )
# df["label"] = y.numpy()  # добавляем колонку "label"


# df.to_csv("data.csv", index=False)
# dataset.add_files("data.csv")
# dataset.upload()
# dataset.finalize()

ClearML results page: https://app.clear.ml/projects/88d3825bfe834623bac1f1120171a18e/experiments/be85bd75cc1d4f7d9a7fad373031d669/output/log
ClearML dataset page: https://app.clear.ml/datasets/simple/88d3825bfe834623bac1f1120171a18e/experiments/be85bd75cc1d4f7d9a7fad373031d669
Uploading dataset changes (1 files compressed to 245 B) to https://files.clear.ml
File compression and upload completed: total size 245 B, 1 chunk(s) stored (average size 245 B)


True